goal : extract dinov3 features from MedSegBench

In [123]:
from src.config import load_config
config = load_config()
from medsegbench import AbdomenUSMSBench
from torchvision import transforms

# Define a custom transform
to_tensor = transforms.Compose([
    transforms.Resize((224, 224)),  # Resizes images to 224x224
    transforms.ToTensor(),  # Converts PIL to tensor [C, H, W] in [0, 1]
    # resizes images to 256x256
])
train_dataset = AbdomenUSMSBench(split='train',
                                root = config["paths"]["medsegbench"],
                                download=False,
                                transform=to_tensor  # This will convert PIL images to tensors
)

In [107]:
from transformers import AutoImageProcessor, AutoModel
from huggingface_hub import login
pretrained_model_name = config["paths"]["ckpts"]["dino_vit"]

#pretrained_model_name = "facebook/dinov3-convnext-large-pretrain-lvd1689m"

processor = AutoImageProcessor.from_pretrained(pretrained_model_name)
model = AutoModel.from_pretrained(
    pretrained_model_name, 
    device_map="auto", 
)



In [108]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [124]:
print(train_dataset[0][0])
# convert PIL to tensor and add batch dimension
images = train_dataset[0][0]
# processor downsizes to a multiple of 16 
inputs = processor(images=images, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model(**inputs)


tensor([[[0.0392, 0.0392, 0.0392,  ..., 0.0392, 0.0392, 0.0392],
         [0.0392, 0.0392, 0.0392,  ..., 0.0392, 0.0392, 0.0392],
         [0.0392, 0.0392, 0.0392,  ..., 0.0392, 0.0392, 0.0392],
         ...,
         [0.0392, 0.0392, 0.0392,  ..., 0.0392, 0.0392, 0.0392],
         [0.0392, 0.0392, 0.0392,  ..., 0.0392, 0.0392, 0.0392],
         [0.0392, 0.0392, 0.0392,  ..., 0.0392, 0.0392, 0.0392]]])


In [125]:
print(images.shape)
print(inputs['pixel_values'].shape)
print("Last hidden state shape:", outputs.last_hidden_state.shape)
# 1 CLS token, 4 register tokens, 196 patch tokens (14x14)


torch.Size([1, 224, 224])
torch.Size([1, 3, 224, 224])
Last hidden state shape: torch.Size([1, 201, 1024])


In [89]:
import torch.nn as nn

class SegmentationHead(nn.Module):
    def __init__(self, embed_dim=1024, num_classes=21, patch_size=16):
        super().__init__()
        self.patch_size = patch_size
        self.num_classes = num_classes
        
        # Simple decoder with upsampling
        self.decoder = nn.Sequential(
            nn.Conv2d(embed_dim, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            
            nn.Conv2d(64, num_classes, kernel_size=1)
        )
    
    def forward(self, patch_features, h, w):
        # Reshape patch tokens to 2D feature map
        # patch_features: [B, num_patches, embed_dim]
        B, N, C = patch_features.shape
        
        # Reshape to spatial grid
        feature_map = patch_features.transpose(1, 2).reshape(B, C, h, w)
        
        # Apply decoder
        output = self.decoder(feature_map)
        
        return output

class DINOv3Segmentation(nn.Module):
    def __init__(self, backbone, num_classes=21, patch_size=16):
        super().__init__()
        self.backbone = backbone
        self.patch_size = patch_size
        
        # Get embedding dimension (1024 for ViT-L)
        embed_dim = backbone.config.hidden_size
        
        self.seg_head = SegmentationHead(
            embed_dim=embed_dim,
            num_classes=num_classes,
            patch_size=patch_size
        )
    
    def forward(self, x):
        # Get image dimensions
        B, C, H, W = x.shape
        
        # Calculate patch grid size
        h = H // self.patch_size
        w = W // self.patch_size
        
        # Extract features from frozen backbone
        with torch.no_grad():
            outputs = self.backbone(x, output_hidden_states=True)
            # Get patch tokens (exclude CLS and register tokens)
            patch_features = outputs.last_hidden_state[:, 5:, :]  # Skip first 5 tokens
        
        # Generate segmentation map
        seg_output = self.seg_head(patch_features, h, w)
        
        return seg_output

In [ ]:
num_classes = 9  # e.g., Pascal VOC
seg_model = DINOv3Segmentation(model, num_classes=num_classes)
seg_model = seg_model.to(device)

# Loss and optimizer (only train decoder)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(seg_model.seg_head.parameters(), lr=1e-4)

# Training loop
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for images, masks in dataloader:

        images = processor(images=images, return_tensors="pt").pixel_values
        images = images.to(device)

        # resize masks to 224x224
        masks = nn.functional.interpolate(masks.unsqueeze(1).float(), size=(224, 224), mode='nearest').squeeze(1).long()
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    
    return total_loss / len(dataloader)

In [132]:
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
train_epoch(seg_model, train_dataloader, criterion, optimizer, device)

Batch Loss: 2.2362101078033447
Batch Loss: 2.1432130336761475
Batch Loss: 2.0048632621765137
Batch Loss: 1.97382652759552
Batch Loss: 1.7935329675674438
Batch Loss: 1.9775066375732422
Batch Loss: 1.7788991928100586
Batch Loss: 1.7504512071609497
Batch Loss: 1.8264670372009277
Batch Loss: 1.8799282312393188
Batch Loss: 1.8632960319519043
Batch Loss: 1.6540884971618652
Batch Loss: 1.7152812480926514
Batch Loss: 1.6109700202941895
Batch Loss: 1.6482374668121338
Batch Loss: 1.6314363479614258
Batch Loss: 1.4548251628875732
Batch Loss: 1.8495351076126099
Batch Loss: 1.6494200229644775
Batch Loss: 1.5504131317138672
Batch Loss: 1.5180034637451172
Batch Loss: 1.6920770406723022
Batch Loss: 1.4687118530273438
Batch Loss: 1.6073014736175537
Batch Loss: 1.6302491426467896
Batch Loss: 1.7200216054916382
Batch Loss: 1.4731194972991943
Batch Loss: 1.470137357711792
Batch Loss: 1.6570632457733154
Batch Loss: 1.400325894355774
Batch Loss: 1.4071483612060547
Batch Loss: 1.412832260131836
Batch Loss: 1

1.5136175718572404

In [ ]:
val_dataset = AbdomenUSMSBench(split='val',
                              root = config["paths"]["medsegbench"],
                              download=False,
                              transform=to_tensor  # This will convert PIL images to tensors
)

def compute_dice(pred, target, num_classes, ignore_background=True):
    """Compute dice coefficient for multi-class segmentation"""
    pred = pred.argmax(dim=1)  # [B, H, W]
    dice_scores = []
    
    start_class = 1 if ignore_background else 0
    for c in range(start_class, num_classes):
        pred_c = (pred == c).float()
        target_c = (target == c).float()
        
        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()
        
        if union == 0:
            continue    
        
        dice = (2.0 * intersection) / (union + 1e-8)
        dice_scores.append(dice.item())
    
    return sum(dice_scores) / len(dice_scores) if dice_scores else 0.0

def val_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_dice = 0
    
    with torch.no_grad():
        for images, masks in dataloader:
            images = processor(images=images, return_tensors="pt").pixel_values
            images = images.to(device)

            # resize masks to 224x224
            masks = nn.functional.interpolate(masks.unsqueeze(1).float(), size=(224, 224), mode='nearest').squeeze(1).long()
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            dice = compute_dice(outputs, masks, num_classes=num_classes)
            
            total_loss += loss.item()
            total_dice += dice
    
    return total_loss / len(dataloader), total_dice / len(dataloader)

In [134]:
for epoch in range(5):
    train_loss = train_epoch(seg_model, train_dataloader, criterion, optimizer, device)
    val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False)
    val_loss, val_dice = val_epoch(seg_model, val_dataloader, criterion, device)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

Batch Loss: 1.3583494424819946
Batch Loss: 1.1841174364089966
Batch Loss: 1.236568570137024
Batch Loss: 1.3036530017852783
Batch Loss: 1.3435927629470825
Batch Loss: 1.4852113723754883
Batch Loss: 1.1240307092666626
Batch Loss: 1.2421339750289917
Batch Loss: 1.1481064558029175
Batch Loss: 1.2682061195373535
Batch Loss: 1.2821298837661743
Batch Loss: 1.2011137008666992
Batch Loss: 1.2019877433776855
Batch Loss: 1.2258456945419312
Batch Loss: 1.3021728992462158
Batch Loss: 1.1484719514846802
Batch Loss: 1.2368183135986328
Batch Loss: 1.1419620513916016
Batch Loss: 1.1614512205123901
Batch Loss: 1.1864495277404785
Batch Loss: 1.1244323253631592
Batch Loss: 1.473885416984558
Batch Loss: 1.420952320098877
Batch Loss: 1.2233237028121948
Batch Loss: 1.1232426166534424
Batch Loss: 1.296419620513916
Batch Loss: 1.2004626989364624
Batch Loss: 1.171311616897583
Batch Loss: 1.1164464950561523
Batch Loss: 1.0966843366622925
Batch Loss: 1.2797327041625977
Batch Loss: 1.1161720752716064
Batch Loss: 1